In [1]:
from pathlib import Path
import os
import sys
from torchvision.datasets import VOCDetection
import torch
from torch.utils.data import DataLoader

In [2]:
ROOT_DIR = Path.cwd().parent
ROOT_DIR

PosixPath('/home/chiendoan/Code_Module6/YOU_ONLY_LOOK_ONCE_YOLO_V1_V2')

In [3]:
data_dir = ROOT_DIR/'data'/'train'
train_voc = VOCDetection(
    root=str(data_dir),
    year='2012',
    image_set='train',
    download=True
)

In [4]:
train_voc[0]

(<PIL.Image.Image image mode=RGB size=500x442>,
 {'annotation': {'folder': 'VOC2012',
   'filename': '2008_000008.jpg',
   'source': {'database': 'The VOC2008 Database',
    'annotation': 'PASCAL VOC2008',
    'image': 'flickr'},
   'size': {'width': '500', 'height': '442', 'depth': '3'},
   'segmented': '0',
   'object': [{'name': 'horse',
     'pose': 'Left',
     'truncated': '0',
     'occluded': '1',
     'bndbox': {'xmin': '53', 'ymin': '87', 'xmax': '471', 'ymax': '420'},
     'difficult': '0'},
    {'name': 'person',
     'pose': 'Unspecified',
     'truncated': '1',
     'occluded': '0',
     'bndbox': {'xmin': '158', 'ymin': '44', 'xmax': '289', 'ymax': '167'},
     'difficult': '0'}]}})

In [5]:
import timm

/home/chiendoan/Code_Module6/YOU_ONLY_LOOK_ONCE_YOLO_V1_V2/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
model = timm.create_model('resnet18.a3_in1k', pretrained=True)

In [7]:
for name, module in model.named_modules():
    if hasattr(module, 'parameters'):
        istrainable = any(param.requires_grad for param in module.parameters())
        print(f'name: {name} - {'trainable' if istrainable else 'not_trainable'}')

name:  - trainable
name: conv1 - trainable
name: bn1 - trainable
name: act1 - not_trainable
name: maxpool - not_trainable
name: layer1 - trainable
name: layer1.0 - trainable
name: layer1.0.conv1 - trainable
name: layer1.0.bn1 - trainable
name: layer1.0.drop_block - not_trainable
name: layer1.0.act1 - not_trainable
name: layer1.0.aa - not_trainable
name: layer1.0.conv2 - trainable
name: layer1.0.bn2 - trainable
name: layer1.0.act2 - not_trainable
name: layer1.1 - trainable
name: layer1.1.conv1 - trainable
name: layer1.1.bn1 - trainable
name: layer1.1.drop_block - not_trainable
name: layer1.1.act1 - not_trainable
name: layer1.1.aa - not_trainable
name: layer1.1.conv2 - trainable
name: layer1.1.bn2 - trainable
name: layer1.1.act2 - not_trainable
name: layer2 - trainable
name: layer2.0 - trainable
name: layer2.0.conv1 - trainable
name: layer2.0.bn1 - trainable
name: layer2.0.drop_block - not_trainable
name: layer2.0.act1 - not_trainable
name: layer2.0.aa - not_trainable
name: layer2.0.conv

In [8]:
model.layer4

Sequential(
  (0): BasicBlock(
    (conv1): Conv2d(256, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (drop_block): Identity()
    (act1): ReLU(inplace=True)
    (aa): Identity()
    (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (act2): ReLU(inplace=True)
    (downsample): Sequential(
      (0): Conv2d(256, 512, kernel_size=(1, 1), stride=(2, 2), bias=False)
      (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
  )
  (1): BasicBlock(
    (conv1): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (drop_block): Identity()
   

In [9]:
for name, module in model.layer4[0].named_modules():
    if isinstance(module, torch.nn.Conv2d):
        print(name, module.stride)

conv1 (2, 2)
conv2 (1, 1)
downsample.0 (2, 2)


# Test Src Orgainize data

In [10]:
SRC_DIR = ROOT_DIR/'src'
sys.path.append(str(ROOT_DIR))
sys.path

['/home/chiendoan/miniconda3/lib/python314.zip',
 '/home/chiendoan/miniconda3/lib/python3.14',
 '/home/chiendoan/miniconda3/lib/python3.14/lib-dynload',
 '',
 '/home/chiendoan/Code_Module6/YOU_ONLY_LOOK_ONCE_YOLO_V1_V2/.venv/lib/python3.14/site-packages',
 '/home/chiendoan/Code_Module6/YOU_ONLY_LOOK_ONCE_YOLO_V1_V2']

In [11]:
from src.data.orgainize_data import *

In [12]:
from functools import partial

In [13]:
train_dir = Path(ROOT_DIR/'data'/'train'/'VOCdevkit'/'VOC2012'/'ImageSets'/'Main')
# Using set datatype to collect single value
CLASSES = sorted({
    f.rsplit('_',1)[0]
    for f in os.listdir(train_dir)
    if '_' in f and f.endswith('.txt')
})
CLASSES

['aeroplane',
 'bicycle',
 'bird',
 'boat',
 'bottle',
 'bus',
 'car',
 'cat',
 'chair',
 'cow',
 'diningtable',
 'dog',
 'horse',
 'motorbike',
 'person',
 'pottedplant',
 'sheep',
 'sofa',
 'train',
 'tvmonitor']

In [14]:
CLASS2INDEX = { clss: (index+1) for index, clss in enumerate(CLASSES)}
CLASS2INDEX

{'aeroplane': 1,
 'bicycle': 2,
 'bird': 3,
 'boat': 4,
 'bottle': 5,
 'bus': 6,
 'car': 7,
 'cat': 8,
 'chair': 9,
 'cow': 10,
 'diningtable': 11,
 'dog': 12,
 'horse': 13,
 'motorbike': 14,
 'person': 15,
 'pottedplant': 16,
 'sheep': 17,
 'sofa': 18,
 'train': 19,
 'tvmonitor': 20}

In [15]:
dataset = VOCdataset(
    data_dir=str(ROOT_DIR/'data'/'train'),
    class2idx=CLASS2INDEX
)

In [16]:
dataset[0][1]['boxes']

tensor([[209.0000, 166.5000, 418.0000, 333.0000],
        [ 65.5000,  61.5000, 131.0000, 123.0000]])

In [17]:
data_cluster = dataset[0][1]['boxes'].clone()[:,2:]
data_cluster

tensor([[418., 333.],
        [131., 123.]])

In [18]:
for i in range(1, dataset.__len__()):
    data_cluster = torch.cat([data_cluster, dataset[i][1]['boxes'].clone()[:,2:]], dim=0)

In [19]:
data_cluster.shape

torch.Size([15774, 2])

In [20]:
type(data_cluster)

torch.Tensor

In [21]:
from model.Kmean_YOLOv2 import *

In [22]:
Kmean_cluster = Kmean()

In [23]:
Kmean_cluster(data_cluster)

tensor([[ 33.9074,  43.1555],
        [ 94.8374, 118.0962],
        [159.8270, 256.6951],
        [314.6092, 157.3282],
        [375.2903, 333.2395]])